<a href="https://colab.research.google.com/github/zydanne-costa/Ondas_ADCP_SCO_Mar_Nov_2025/blob/main/Resultados_PSD_1D_CHU_SEC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este notebook realiza a análise espectral das ondas (PSD) utilizando o método de Welch aplicado aos dados STrk do ADCP, comparando campanhas em períodos chuvosos e menos chuvosos.

O produto final é o plot da figura de psd das duas campanhas indicando o pico de frequência de cada uma.

In [ ]:
# ============================================================
# ANÁLISE ESPECTRAL DAS ONDAS (PSD)
# Método de Welch aplicado aos dados STrk do ADCP
# Dissertação de Mestrado
# ============================================================

# ============================================================
# Bibliotecas
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from scipy.signal import welch
from scipy.signal import detrend

from google.colab import drive

In [ ]:
# ============================================================
# Montar Google Drive
# ============================================================

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# Diretórios dos dados
# ============================================================

dir_chuvoso = "/content/drive/MyDrive/Ondas/Dados/marco2025/SCO1/STrk"

dir_seco = "/content/drive/MyDrive/Ondas/Dados/novembro2025/SCO2/STrk_2"

dir_saida = "/content/drive/MyDrive/Ondas/Dados/Refinados"

In [ ]:
# ============================================================
# Parâmetros da análise espectral
# ============================================================

# Frequência de amostragem (Hz)
fs = 2.0

# Comprimento dos segmentos do Welch
nperseg = 512

# Sobreposição (50%)
noverlap = 256

# Janela espectral
janela = "hann"

In [ ]:
# ============================================================
# Função para leitura dos arquivos STrk
# ============================================================

def ler_strk(arquivo):
    """
    Lê um arquivo Surface Track (STrk) do ADCP.

    Parâmetros
    ----------
    arquivo : str
        Caminho do arquivo STrk.

    Retorna
    -------
    pandas.DataFrame
        DataFrame contendo os quatro feixes (Beam 1 a Beam 4)
        em metros, com valores inválidos removidos e pequenas
        lacunas interpoladas.
    """

    # Leitura do arquivo
    df = pd.read_csv(
        arquivo,
        sep=r"\s+",
        skiprows=16,
        header=None
    )

    # Mantém apenas os quatro primeiros feixes
    df = df.iloc[:, :4]

    df.columns = [
        "Beam 1",
        "Beam 2",
        "Beam 3",
        "Beam 4"
    ]

    # Remove valores inválidos
    df.replace([-32768, 0], np.nan, inplace=True)

    # Conversão de milímetros para metros
    df = df / 1000.0

    # Interpolação de pequenas lacunas
    df.interpolate(
        method="linear",
        limit=4,
        inplace=True
    )

    return df

In [ ]:
# ============================================================
# Função para cálculo da PSD pelo método de Welch
# ============================================================

def calcular_psd_welch(df,
                       fs=2.0,
                       nperseg=512,
                       noverlap=256):
    """
    Calcula a densidade espectral de potência (PSD) utilizando
    o método de Welch para cada beam do ADCP.

    Além da PSD individual de cada beam, calcula também a
    PSD média dos quatro feixes e os principais parâmetros
    espectrais.

    Parâmetros
    ----------
    df : pandas.DataFrame
        DataFrame contendo os quatro beams.

    fs : float
        Frequência de amostragem (Hz).

    nperseg : int
        Comprimento dos segmentos utilizados pelo método
        de Welch.

    noverlap : int
        Número de amostras de sobreposição entre segmentos.

    Retorna
    -------
    dict
        Dicionário contendo a PSD de cada beam e a PSD média.
    """

    resultados = {}

    # ========================================================
    # Processamento individual dos quatro beams
    # ========================================================

    for beam in df.columns:

        # Série temporal
        serie = df[beam].dropna().values

        # Garante número mínimo de amostras
        if len(serie) < nperseg:
            continue

        # Remove tendência linear
        serie = detrend(serie)

        # Método de Welch
        frequencia, psd = welch(
            serie,
            fs=fs,
            window="hann",
            nperseg=nperseg,
            noverlap=noverlap,
            detrend=False,
            scaling="density"
        )

        # Momento espectral de ordem zero
        m0 = np.trapezoid(psd, frequencia)

        # Altura significativa espectral
        hm0 = 4 * np.sqrt(m0)

        # Frequência de pico
        fp = frequencia[np.argmax(psd)]

        # Período de pico
        tp = 1 / fp if fp > 0 else np.nan

        resultados[beam] = {

            "frequencia": frequencia,

            "psd": psd,

            "m0": m0,

            "Hm0": hm0,

            "fp": fp,

            "Tp": tp

        }

    # ========================================================
    # PSD média dos quatro beams
    # ========================================================

    psd_media = np.mean(

        [
            resultados["Beam 1"]["psd"],
            resultados["Beam 2"]["psd"],
            resultados["Beam 3"]["psd"],
            resultados["Beam 4"]["psd"]
        ],

        axis=0

    )

    frequencia = resultados["Beam 1"]["frequencia"]

    m0_media = np.trapezoid(psd_media, frequencia)

    hm0_media = 4 * np.sqrt(m0_media)

    fp_media = frequencia[np.argmax(psd_media)]

    tp_media = 1 / fp_media if fp_media > 0 else np.nan

    resultados["media"] = {

        "frequencia": frequencia,

        "psd": psd_media,

        "m0": m0_media,

        "Hm0": hm0_media,

        "fp": fp_media,

        "Tp": tp_media

    }

    return resultados

In [ ]:
# ============================================================
# Processa todos os arquivos STrk de uma campanha
# ============================================================

def processar_campanha(diretorio):
    """
    Calcula a PSD média de todos os arquivos STrk de uma campanha.

    O Hm0, fp e Tp são calculados a partir do espectro médio
    da campanha.
    """

    arquivos = sorted(glob.glob(os.path.join(diretorio, "*.txt")))

    lista_psd = []

    frequencia = None

    print(f"{len(arquivos)} arquivos encontrados.")

    for arquivo in arquivos:

        try:

            df = ler_strk(arquivo)

            resultado = calcular_psd_welch(df)

            lista_psd.append(resultado["media"]["psd"])

            frequencia = resultado["media"]["frequencia"]


        except Exception as erro:

            print(f"Erro em {os.path.basename(arquivo)}")
            print(erro)


    # PSD média da campanha
    psd_media_campanha = np.mean(lista_psd, axis=0)


    # Momento espectral zero
    m0 = np.trapezoid(
        psd_media_campanha,
        frequencia
    )


    # Altura significativa espectral
    hm0 = 4 * np.sqrt(m0)


    # Frequência de pico
    fp = frequencia[
        np.argmax(psd_media_campanha)
    ]


    # Período de pico
    tp = 1 / fp if fp > 0 else np.nan


    return {

        "frequencia": frequencia,

        "psd": psd_media_campanha,

        "Hm0": hm0,

        "fp": fp,

        "Tp": tp

    }

In [ ]:
# ============================================================
# Calcula a PSD média das campanhas
# ============================================================

campanha_chuvoso = processar_campanha(dir_chuvoso)

print()

campanha_seco = processar_campanha(dir_seco)

442 arquivos encontrados.

169 arquivos encontrados.


In [ ]:
print("========== CHUVOSO ==========")

print(f"Hm0 = {campanha_chuvoso['Hm0']:.3f} m")

print(f"fp  = {campanha_chuvoso['fp']:.4f} Hz")

print(f"Tp  = {campanha_chuvoso['Tp']:.2f} s")

print()

print("========== MENOS CHUVOSO ==========")

print(f"Hm0 = {campanha_seco['Hm0']:.3f} m")

print(f"fp  = {campanha_seco['fp']:.4f} Hz")

print(f"Tp  = {campanha_seco['Tp']:.2f} s")

========== CHUVOSO ==========
Hm0 = 0.825 m
fp  = 0.1250 Hz
Tp  = 8.00 s

========== MENOS CHUVOSO ==========
Hm0 = 0.893 m
fp  = 0.1953 Hz
Tp  = 5.12 s


In [ ]:
# Set global font to serif
plt.rcParams['font.family'] = 'serif'

# Calculate peak frequencies and corresponding PSD values
fp_chuvoso = campanha_chuvoso["fp"]
idx_chuvoso = np.argmin(np.abs(campanha_chuvoso["frequencia"] - fp_chuvoso))
psd_chuvoso_at_fp = campanha_chuvoso["psd"][idx_chuvoso]

fp_seco_main = campanha_seco["fp"]
idx_seco_main = np.argmin(np.abs(campanha_seco["frequencia"] - fp_seco_main))
psd_seco_at_fp_main = campanha_seco["psd"][idx_seco_main]

# For the second peak of "Menos Chuvoso" (hardcoded to 0.223)
fp_seco_secondary = 0.223
idx_seco_secondary = np.argmin(np.abs(campanha_seco["frequencia"] - fp_seco_secondary))
psd_seco_at_fp_secondary = campanha_seco["psd"][idx_seco_secondary]

plt.figure(figsize=(11,5))

plt.plot(
    campanha_chuvoso["frequencia"],
    campanha_chuvoso["psd"],
    label="Chuvoso",
    color='darkblue'
)

plt.plot(
    campanha_seco["frequencia"],
    campanha_seco["psd"],
    label="Menos chuvoso",
    color='orangered'
)

# Marcação dos picos espectrais
# Chuvoso
plt.vlines(
    fp_chuvoso,
    0, # Start from y=0
    psd_chuvoso_at_fp, # End at the PSD value
    linestyle=":",
    color='mediumblue',
    label=f"fp Chuvoso = {fp_chuvoso:.3f}".replace('.', ',') + " Hz"
)

# Menos Chuvoso - Principal peak
plt.vlines(
    fp_seco_main,
    0, # Start from y=0
    psd_seco_at_fp_main, # End at the PSD value
    linestyle=":",
    linewidth=1.5,
    color='tomato',
    label=f"fp Menos Chuvoso = {fp_seco_main:.3f}".replace('.', ',') + " Hz"
)

# Menos Chuvoso - Secondary peak
plt.vlines(
    fp_seco_secondary,
    0, # Start from y=0
    psd_seco_at_fp_secondary, # End at the PSD value
    linestyle=":",
    linewidth=1.5,
    color='tomato',
    label=f"fp Menos Chuvoso = {fp_seco_secondary:.3f}".replace('.', ',') + " Hz"
)

plt.xlabel("Frequência (Hz)")
plt.ylabel("PSD (m²/Hz)")

plt.title(
    "Densidade espectral de potência em diferentes períodos"
)

plt.legend()

plt.grid(True, linestyle=':', alpha=0.6) # Discreet grid

# Format axis ticks to use comma as decimal separator
formatter = mticker.FormatStrFormatter('%.2f')
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: formatter(x).replace('.', ',')))
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: formatter(x).replace('.', ',')))

plt.xlim(0,0.6)
plt.ylim(0.02,0.17)

# Save the plot with high resolution
plt.savefig(f"{dir_saida}/PSD_1D_periodos.png", dpi=300, bbox_inches='tight')

plt.show()